# 🌿 Medicinal Plant Recognition Model
**EfficientNetB3 Transfer Learning on IMFI Indian Medicinal Flower Dataset (28 Classes)**

---

### Steps:
1. Upload your dataset ZIP to Google Drive
2. Mount Drive and extract
3. Train the model (GPU accelerated)
4. Download the trained model files

> ⚠️ Make sure to set **Runtime → Change runtime type → T4 GPU** before starting!

## Step 1: Mount Google Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Upload & Extract Dataset

**Option A**: If your dataset is already in Google Drive as a ZIP file, update the path below:

**Option B**: If you want to upload directly, use the file upload cell instead.

In [ ]:
import os
import zipfile

# ============================================================
# OPTION A: Dataset is in Google Drive as a ZIP
# Update this path to where your ZIP file is in Drive
# ============================================================
DRIVE_ZIP_PATH = "/content/drive/MyDrive/IMFI_Dataset.zip"  # <-- CHANGE THIS

EXTRACT_DIR = "/content/dataset"

if os.path.exists(DRIVE_ZIP_PATH):
    print(f"Extracting {DRIVE_ZIP_PATH}...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print("Done!")
else:
    print(f"ZIP not found at {DRIVE_ZIP_PATH}")
    print("Either update the path or use Option B (next cell) to upload directly.")

In [ ]:
# ============================================================
# OPTION B: Upload ZIP directly from your computer
# (Skip this if you used Option A)
# ============================================================
from google.colab import files
import zipfile, os

print("Upload your dataset ZIP file:")
uploaded = files.upload()

EXTRACT_DIR = "/content/dataset"
for fname in uploaded.keys():
    print(f"Extracting {fname}...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print("Done!")

In [ ]:
# ============================================================
# Find the actual dataset directory (with class subfolders)
# ============================================================
import os

def find_dataset_root(base_dir):
    """Walk down until we find a directory containing class subfolders with images."""
    for root, dirs, files_list in os.walk(base_dir):
        # Check if this directory has subdirectories containing image files
        if len(dirs) >= 10:  # At least 10 class folders
            sample_dir = os.path.join(root, dirs[0])
            if os.path.isdir(sample_dir):
                sample_files = os.listdir(sample_dir)
                img_files = [f for f in sample_files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                if len(img_files) > 0:
                    return root
    return base_dir

DATA_DIR = find_dataset_root(EXTRACT_DIR)
classes = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f"Dataset root: {DATA_DIR}")
print(f"Found {len(classes)} classes:")
for i, c in enumerate(classes, 1):
    count = len(os.listdir(os.path.join(DATA_DIR, c)))
    print(f"  {i:2d}. {c} ({count} images)")

## Step 3: Load & Prepare Data

In [ ]:
import numpy as np
import pandas as pd
import json
from sklearn.model_selection import train_test_split

# Build DataFrame of file paths and labels
filepaths = []
labels = []
for folder in sorted(os.listdir(DATA_DIR)):
    folder_path = os.path.join(DATA_DIR, folder)
    if not os.path.isdir(folder_path):
        continue
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        if os.path.isfile(fpath):
            filepaths.append(fpath)
            labels.append(folder)

df = pd.DataFrame({"filepaths": filepaths, "labels": labels})
print(f"Total images: {len(df)}")
print(f"Classes: {df['labels'].nunique()}")
print(f"\nSamples per class:")
print(df['labels'].value_counts().to_string())

In [ ]:
# Split: 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(df, train_size=0.8, shuffle=True, random_state=42, stratify=df['labels'])
valid_df, test_df = train_test_split(temp_df, train_size=0.5, shuffle=True, random_state=42, stratify=temp_df['labels'])

print(f"Train: {len(train_df)} | Validation: {len(valid_df)} | Test: {len(test_df)}")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32  # Larger batch on GPU

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepaths', y_col='labels',
    target_size=IMG_SIZE, class_mode='categorical',
    batch_size=BATCH_SIZE, shuffle=True
)

valid_gen = val_datagen.flow_from_dataframe(
    valid_df, x_col='filepaths', y_col='labels',
    target_size=IMG_SIZE, class_mode='categorical',
    batch_size=BATCH_SIZE, shuffle=False
)

test_gen = val_datagen.flow_from_dataframe(
    test_df, x_col='filepaths', y_col='labels',
    target_size=IMG_SIZE, class_mode='categorical',
    batch_size=BATCH_SIZE, shuffle=False
)

# Save class indices
class_indices = train_gen.class_indices
with open('medicinal_class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=2)

num_classes = len(class_indices)
print(f"\nNumber of classes: {num_classes}")
print(f"Class indices saved to medicinal_class_indices.json")

## Step 4: Build EfficientNetB3 Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Build model
img_shape = (224, 224, 3)
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=img_shape)
base_model.trainable = False  # Freeze base initially

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)

x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(64, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=Adamax(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\nTotal params: {model.count_params():,}")
trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
print(f"Trainable params: {trainable:,}")

## Step 5: Train — Phase 1 (Frozen Base)

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('medicinal_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

INITIAL_EPOCHS = 10

print("Phase 1: Training classification head (base frozen)...")
history1 = model.fit(
    train_gen,
    epochs=INITIAL_EPOCHS,
    validation_data=valid_gen,
    callbacks=callbacks,
    verbose=1
)

## Step 6: Train — Phase 2 (Fine-tune Top Layers)

In [ ]:
# Unfreeze the top 30 layers of the base model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adamax(learning_rate=0.0001),  # Lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

FINE_TUNE_EPOCHS = 10

print("Phase 2: Fine-tuning top layers...")
history2 = model.fit(
    train_gen,
    epochs=FINE_TUNE_EPOCHS,
    validation_data=valid_gen,
    callbacks=callbacks,
    verbose=1
)

## Step 7: Evaluate on Test Set

In [ ]:
print("Evaluating on test set...")
test_loss, test_accuracy = model.evaluate(test_gen, verbose=1)
print(f"\n{'='*40}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Test Loss:     {test_loss:.4f}")
print(f"{'='*40}")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Combine histories
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
epochs_range = range(1, len(acc) + 1)

ax1.plot(epochs_range, acc, 'b-', label='Train Accuracy')
ax1.plot(epochs_range, val_acc, 'r-', label='Val Accuracy')
ax1.axvline(x=len(history1.history['accuracy']), color='g', linestyle='--', label='Fine-tune start')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, loss, 'b-', label='Train Loss')
ax2.plot(epochs_range, val_loss, 'r-', label='Val Loss')
ax2.axvline(x=len(history1.history['loss']), color='g', linestyle='--', label='Fine-tune start')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## Step 8: Save & Download Model Files

In [ ]:
# Save the final model
model.save('medicinal_model.keras')
print("Model saved: medicinal_model.keras")
print("Class indices saved: medicinal_class_indices.json")
print(f"\nModel size: {os.path.getsize('medicinal_model.keras') / (1024*1024):.1f} MB")

In [ ]:
# Download the files to your computer
from google.colab import files

print("Downloading model files...")
files.download('medicinal_model.keras')
files.download('medicinal_class_indices.json')
print("\nDone! Place these files in your 'Mini Project' folder, then run:")
print("  python test_model.py")

## (Optional) Quick Test on Colab

In [ ]:
# Upload a test image and predict
from google.colab import files
from tensorflow.keras.preprocessing import image
import numpy as np

print("Upload a flower/leaf image to test:")
uploaded = files.upload()

# Load class indices
with open('medicinal_class_indices.json', 'r') as f:
    class_indices = json.load(f)
labels = {v: k for k, v in class_indices.items()}

for fname in uploaded.keys():
    img = image.load_img(fname, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)
    top3 = np.argsort(predictions[0])[::-1][:3]

    print(f"\nResults for {fname}:")
    print("=" * 40)
    for rank, idx in enumerate(top3, 1):
        name = labels.get(idx, 'Unknown')
        conf = predictions[0][idx]
        print(f"  {rank}. {name}: {conf:.2%}")

    plt.imshow(image.load_img(fname))
    plt.title(f"{labels[top3[0]]} ({predictions[0][top3[0]]:.1%})")
    plt.axis('off')
    plt.show()